<a href="https://colab.research.google.com/github/darieldatoon/hse-agent-workshop/blob/main/notebooks/00_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 00 · Setup

**Run this before the lab.** It checks that your LangSmith key works, that model calls go
through, and that traces land in **your own workspace**.

You need:

- A **personal Google account**, to run Colab.
- Your **LangSmith invite** accepted. It went to your work email.
- The **workshop key**, from your instructor.

---

## Step 1 — Find your workspace

Open [smith.langchain.com](https://smith.langchain.com). Use the workspace switcher at the top
left and pick the workspace with **your name**. Note its name exactly as shown.

---

## Step 2 — Store the workshop key in Colab Secrets

1. In the **left sidebar of this notebook**, click the 🔑 key icon.
2. Click **Add new secret**.
3. Name it **exactly** `LANGSMITH_API_KEY`.
4. Paste the workshop key into **Value**.
5. **Turn on "Notebook access".**

### Running locally instead?

From a clone of this repo, `uv sync`, copy `.env.example` to `.env`, paste the key, then
`uv run jupyter lab notebooks`. The setup cell reads `.env` when there is no Colab.

---

## Step 3 — Run the setup cell

Type your workspace name in the **WORKSPACE** field, then run the cell.

In [ ]:
# --- snippet:setup v1 ---
import os

import requests

WORKSPACE = ""  #@param {type:"string"}

try:
    from google.colab import userdata

    %pip install -qq --progress-bar off \
      "langchain~=1.4.3" \
      "langchain-openai~=1.6.7" \
      "langsmith~=0.14.4" \
      "agentevals~=0.0.9"

    os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")
except ImportError:
    from dotenv import load_dotenv

    load_dotenv()

key = os.environ["LANGSMITH_API_KEY"]
os.environ["LANGSMITH_GATEWAY_API_KEY"] = os.environ.get("LANGSMITH_GATEWAY_API_KEY") or key
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "hse-00-setup"

# The workshop key reaches every attendee's workspace: without a workspace, your traces,
# datasets and experiments land in a shared one and collide with everyone else's.
if WORKSPACE:
    endpoint = os.environ.get("LANGSMITH_ENDPOINT", "https://api.smith.langchain.com")
    reply = requests.get(f"{endpoint}/api/v1/workspaces", headers={"x-api-key": key}, timeout=30)
    reply.raise_for_status()
    wanted = WORKSPACE.strip().casefold()
    matches = [w["id"] for w in reply.json() if w["display_name"].casefold() == wanted]
    if len(matches) != 1:
        raise ValueError(f"No workspace named {WORKSPACE!r}. Type it exactly as LangSmith shows it.")
    os.environ["LANGSMITH_WORKSPACE_ID"] = matches[0]
elif not os.environ.get("LANGSMITH_WORKSPACE_ID"):
    raise ValueError("Type your workspace name in the WORKSPACE field, then run this cell again.")

MODEL = "langsmith:openai/gpt-5.6-luna"
# --- /snippet ---

print("Ready.")

---

## Step 4 — Test a model call

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(MODEL)

response = model.invoke("Return the text 'Hello, World!'.")
response.text

---

## Step 5 — Check the trace landed in your workspace

In [ ]:
import time

from langsmith import Client

client = Client()
for _ in range(10):
    runs = list(client.list_runs(project_name=os.environ["LANGSMITH_PROJECT"], limit=1))
    if runs:
        break
    time.sleep(3)
else:
    raise RuntimeError("No trace yet. Wait a few seconds and run this cell again.")

print("Trace found:", client.get_run_url(run=runs[0]))

---

## ➡️ Next

**[01 · Find the bugs, then prove the fix](https://colab.research.google.com/github/darieldatoon/hse-agent-workshop/blob/main/notebooks/01_find_and_fix.ipynb)**